# Amul Reconciliation — Redash / Trino Data Consolidation



Pulls two invoice extracts from Trino and merges them into **one consolidated Parquet file**

built for large volumes (chunked SQL reads streamed directly to Parquet — data is never fully

held in memory).



| Source | Entity | Query | Date placeholders |

|--------|--------|-------|-------------------|

| **HPTech** | Zomato Hyperpure | HP WMS GRN extract | `{{GRN from}}` / `{{GRN to}}` |

| **GPOS** | Blinkit | Document-digitisation variance extract | `{{start_date}}` / `{{end_date}}` |



**How to run:** edit the dates in the CONFIG cell, then run every cell top-to-bottom.

See `PROJECT_INSTRUCTIONS.md` for the full spec, connection setup, and schema notes.

## 1. CONFIG — edit dates, then run everything top-to-bottom

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import os

# ---- Query 1 (HPTech / Zomato Hyperpure): grn_completed_on window ----------
GRN_FROM   = '2025-06-01'
GRN_TO     = '2025-06-30'

# ---- Query 2 (GPOS / Blinkit): updated_at + grn_date window ----------------
START_DATE = '2025-06-01'
END_DATE   = '2025-06-30'

# ---- Output ---------------------------------------------------------------
# Set OUT_DIR to any path YOU can write to. Default tries ~/amul_recon, then
# ./amul_recon (next to this notebook). '/home/Documents' is NOT writable here.
OUT_DIR = os.path.expanduser('~/amul_recon')


def _ensure_writable(preferred):
    """Return the first directory we can actually create + write into."""
    candidates = [preferred, os.path.join(os.getcwd(), 'amul_recon')]
    for cand in candidates:
        try:
            os.makedirs(cand, exist_ok=True)
            probe = os.path.join(cand, '.write_test')
            with open(probe, 'w') as f:
                f.write('ok')
            os.remove(probe)
            return cand
        except (PermissionError, OSError) as e:
            print(f'  not writable: {cand}  ({e})')
    raise PermissionError(
        'No writable output dir. Set OUT_DIR to a path you own and re-run this cell.'
    )


OUT_DIR  = _ensure_writable(OUT_DIR)
OUT_FILE = os.path.join(OUT_DIR, 'amul_invoice_extract.parquet')   # consolidated (both sources)
CSV_FILE = os.path.join(OUT_DIR, 'amul_invoice_extract.csv')

# ---- Performance knobs ----------------------------------------------------
CHUNKSIZE    = 100_000     # rows pulled per fetch; lower it if memory is tight
WRITE_CSV    = True        # CSV is huge & slow for big extracts; set False to skip
CSV_MAX_ROWS = 2_000_000   # auto-skip CSV above this many rows (parquet still written)

print(f'HPTech (GRN) window : {GRN_FROM} -> {GRN_TO}')
print(f'GPOS window         : {START_DATE} -> {END_DATE}')
print(f'Output dir          : {OUT_DIR}')
print(f'Parquet output      : {OUT_FILE}')
print(f'CSV output          : {CSV_FILE}  (write={WRITE_CSV})')

## 2. Imports, connection & helpers

In [ ]:
# ============================================================================
# Imports + connection + helpers
# ============================================================================
import re
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pencilbox as pb

connection = pb.get_connection('[Warehouse] Trino')


def render(query, **params):
    """Replace Redash-style {{placeholders}} (whitespace-tolerant) with values."""
    out = query
    for key, val in params.items():
        out = re.sub(r'\{\{\s*' + re.escape(key) + r'\s*\}\}', str(val), out)
    leftover = re.findall(r'\{\{.*?\}\}', out)
    if leftover:
        raise ValueError(f'Unsubstituted placeholders remain: {sorted(set(leftover))}')
    return out


def sql(query, label=None):
    """Run a query, return a DataFrame, print row count + elapsed time."""
    t0 = time.time()
    df = pd.read_sql_query(sql=query, con=connection)
    tag = f'[{label}] ' if label else ''
    print(f'  ok {tag}{time.time()-t0:6.1f}s  {len(df):>10,} rows')
    return df

## 3. Consolidated schema & normalisation



The two queries do not return identical columns (GPOS has `vr_id` / `approval_timestamp`;

HPTech has `grn_number`), and a few columns differ in type between sources

(e.g. HPTech emits `asn_quantity` as an empty string). We pin **one explicit target

schema** so every chunk from either source writes into the same Parquet file without

schema drift. `source` marks the origin row-by-row. Identifier columns are kept as

strings; measures as doubles (blanks/`''` -> null); dates/timestamps parsed.

In [ ]:
# ============================================================================
# Target (consolidated) schema — the single source of truth for the output file
# ============================================================================
STRING_COLS = [
    'source', 'entity_name', 'vr_id', 'grn_number', 'invoice_id', 'po_number',
    'vendor_id', 'vendor_name', 'manufacturer_name', 'facility_name', 'city_name',
    'item_id', 'item_name', 'utr_numbers', 'gcmmf_customer_code', 'bank_account',
]
TS_COLS   = ['approval_timestamp']
DATE_COLS = ['invoice_date', 'grn_date']
NUM_COLS  = [
    'invoice_landing_price', 'invoice_quantity', 'final_verified_qty', 'asn_quantity',
    'po_quantity', 'grn_quantity', 'dn_quantity', 'landing_price', 'net_amount',
    'po_landing_price', 'grn_landing_price', 'total_payment_value', 'tds_amount',
]

# Final column order in the consolidated file
COLUMN_ORDER = [
    'source', 'entity_name', 'vr_id', 'approval_timestamp', 'grn_number',
    'invoice_id', 'po_number', 'invoice_date', 'grn_date',
    'vendor_id', 'vendor_name', 'manufacturer_name', 'facility_name', 'city_name',
    'item_id', 'item_name',
    'invoice_landing_price', 'invoice_quantity', 'final_verified_qty', 'asn_quantity',
    'po_quantity', 'grn_quantity', 'dn_quantity', 'landing_price', 'net_amount',
    'po_landing_price', 'grn_landing_price', 'total_payment_value',
    'utr_numbers', 'tds_amount', 'gcmmf_customer_code', 'bank_account',
]

_fields = []
for col in COLUMN_ORDER:
    if col in TS_COLS:
        _fields.append(pa.field(col, pa.timestamp('us')))
    elif col in DATE_COLS:
        _fields.append(pa.field(col, pa.timestamp('us')))
    elif col in NUM_COLS:
        _fields.append(pa.field(col, pa.float64()))
    else:
        _fields.append(pa.field(col, pa.string()))
TARGET_SCHEMA = pa.schema(_fields)


def _dedupe_columns(df):
    """Drop duplicate column names, keeping the first (HP query repeats grn_quantity)."""
    return df.loc[:, ~df.columns.duplicated()]


def normalize(df, source_label):
    """Coerce a raw chunk into the pinned TARGET_SCHEMA (returns a DataFrame)."""
    df = _dedupe_columns(df).copy()
    df['source'] = source_label
    df = df.reindex(columns=COLUMN_ORDER)          # add missing cols as NaN, drop extras
    for c in NUM_COLS:
        df[c] = pd.to_numeric(df[c].replace('', pd.NA), errors='coerce')
    for c in DATE_COLS + TS_COLS:
        df[c] = pd.to_datetime(df[c], errors='coerce')
    for c in STRING_COLS:
        s = df[c].astype('object').where(df[c].notna(), None)
        df[c] = s.map(lambda v: None if v is None else str(v))
    return df


def to_table(df):
    return pa.Table.from_pandas(df, schema=TARGET_SCHEMA, preserve_index=False, safe=False)

## 4. Query 1 — HPTech (Zomato Hyperpure)

Rendered SQL is printed so you can copy-paste it straight into Redash/Trino to sanity-check.

In [ ]:
HP_SQL = r'''SELECT
    'ZOMATO HYPERPURE PRIVATE LIMITED' AS entity_name,
    g.supplier_bill_number AS invoice_id,
    purchase_order_number AS po_number,
    g.grn_number,
    DATE(g.grn_completed_on + INTERVAL '330' MINUTE) AS invoice_date,
    DATE(g.grn_completed_on + INTERVAL '330' MINUTE) AS grn_date,
    po.vendor_id,
    s.outlet_name AS vendor_name,
    '' AS manufacturer_name,
    w.name AS facility_name,
    c.city_name,
    poi.product_number AS item_id,
    poi.product_name AS item_name,
    gi.invoiced_price_per_unit * (1 + gst_rate / 100.0) AS invoice_landing_price,
    gi.invoiced_quantity AS invoice_quantity,
    gi.delivered_quantity AS final_verified_qty,
    '' AS asn_quantity,
    poi.quantity_ordered AS po_quantity,
    gi.grn_quantity,
    CASE
        WHEN gi.grn_quantity IS NOT NULL
             AND gi.invoiced_quantity > gi.grn_quantity
        THEN gi.invoiced_quantity - gi.grn_quantity
        ELSE 0
    END AS dn_quantity,
    grn_price_per_unit * (1 + gst_rate / 100.0) AS landing_price,
    gi.grn_quantity
        * grn_price_per_unit
        * (1 + gst_rate / 100.0) AS net_amount,
    price_per_unit * (1 + gst_rate / 100.0) AS po_landing_price,
    grn_price_per_unit * (1 + gst_rate / 100.0) AS grn_landing_price,
    pop.payment_amount AS total_payment_value,
    pop.transaction_info as utr_numbers,
    pop.payment_amount*0.001 AS tds_amount,
    '' AS gcmmf_customer_code,
    '' AS bank_account
FROM zomato.hp_wms.purchase_order po
LEFT JOIN zomato.hp_wms.purchase_order_items poi
    ON po.id = poi.purchase_order_id
LEFT JOIN zomato.hp_wms.po_grn_item gi
    ON poi.id = gi.purchase_order_item_id
LEFT JOIN zomato.hp_wms.po_grn_mapping g
    ON g.id = gi.po_grn_mapping_id
LEFT JOIN zomato.hp_wms.seller_outlet s
    ON s.id = po.vendor_id
LEFT JOIN zomato.hp_wms.purchase_order_payment pop
    ON pop.id = g.purchase_order_payment_id
LEFT JOIN zomato.hp_wms.warehouse w
    ON po.warehouse_code = w.warehouse_code
LEFT JOIN zomato.hp_wms.city c
    ON w.city_id = c.id
WHERE transaction_id_document_number = zsha1('AAAAG5588Q')
  AND DATE(g.grn_completed_on + INTERVAL '330' MINUTE)
      BETWEEN DATE('{{GRN from}}')
          AND DATE('{{GRN to}}')
  AND g.grn_status = 'COMPLETED'
'''

hp_query = render(HP_SQL, **{'GRN from': GRN_FROM, 'GRN to': GRN_TO})
print(hp_query)

## 5. Query 2 — GPOS (Blinkit)

Uses `{{start_date}}` / `{{end_date}}`. Note the `%%dummy%%` LIKE pattern is kept

double-escaped for the pandas DBAPI paramstyle.

In [ ]:
GPOS_SQL = r'''WITH base_requests AS (
    SELECT
        vr.seller_id,
        vr.id AS request_id,
        vr.verified_invoice_data_id,
        vr.updated_at
    FROM document_digitisation.verification_request vr
    WHERE vr.lake_active_record
      AND vr.request_status = 4
      AND vr.seller_id IN (1,6,9,10)
      AND CAST(vr.updated_at AS date) BETWEEN date('{{start_date}}') AND date('{{end_date}}')
      AND vr.insert_ds_ist >= cast(date('{{start_date}}') - interval '60' day as varchar)
      AND vr.active
),
core_item_data AS (
    SELECT
        br.seller_id,
        ds.name AS seller_name,
        br.request_id,
        br.updated_at,
        UPPER(ivr.invoice_id) AS join_invoice_id,
        ivr.invoice_id,
        ivr.invoice_date,
        ivr.vendor_id,
        ivr.vendor_name,
        ivr.manufacturer_name,
        ivr.facility_name,
        ivr.city_name,
        ivr.grn_date,
        ivr.outlet_id,
        pod.po_number,
        viid.item_id,
        viid.item_name,
        COALESCE(viid.invoice_landing_price, 0) AS invoice_landing_price,
        COALESCE(viid.po_landing_price, 0) AS po_landing_price,
        viid.invoice_quantity,
        COALESCE(
            CASE
                WHEN viid.verified_quantity = 0 THEN viid.invoice_quantity
                ELSE CAST(viid.verified_quantity AS INT)
            END, viid.invoice_quantity
        ) AS final_verified_qty,
        SUM(
            COALESCE(
                CASE
                    WHEN viid.verified_quantity = 0 THEN viid.invoice_quantity
                    ELSE CAST(viid.verified_quantity AS INT)
                END, viid.invoice_quantity
            )
        ) OVER (PARTITION BY ivr.invoice_id, ivr.vendor_id) as invoice_total_verified_qty
    FROM document_digitisation.verified_invoice_items_data viid
    JOIN base_requests br ON br.verified_invoice_data_id = viid.verified_invoice_id
    JOIN document_digitisation.invoice_verification_request ivr ON ivr.request_id = br.request_id
    JOIN document_digitisation.purchase_order_data pod ON pod.request_id = br.request_id
    INNER JOIN document_digitisation.digitised_invoice_item_data diid
        ON diid.request_id = br.request_id AND viid.item_id = diid.item_id
    LEFT JOIN document_digitisation.seller ds ON ds.id = br.seller_id
    JOIN supply_etls.outlet_details od ON od.hot_outlet_id = ivr.outlet_id
    WHERE viid.lake_active_record
      AND viid.item_id NOT LIKE '%%dummy%%'
      AND ivr.active AND ivr.lake_active_record AND ivr.insert_ds_ist >= cast(date('{{start_date}}') - interval '60' day as varchar)
      AND pod.insert_ds_ist >= cast(date('{{start_date}}') - interval '60' day as varchar)
      AND pod.lake_active_record
      AND diid.insert_ds_ist >= cast(date('{{start_date}}') - interval '60' day as varchar)
      AND diid.lake_active_record
      AND CAST(ivr.grn_date AS DATE) >= date('{{start_date}}') - interval '60' day
      AND ivr.manufacturer_name = 'GCMMF'
      AND od.active = 1
      AND od.ars_check = 1
)
,

po_agg as (
select po_number, a.item_id, sum(po_quantity) as po_quantity, sum(grn_quantity) as grn_quantity_purchase_mis, avg(po_landing_price) as po_landing_price, avg(grn_landing_price) as grn_landing_price
from supply_etls.inventory_metrics_purchase_mis a
join supply_etls.item_details b on a.item_id = b.item_id
where
po_to_be_consider = 1
and insert_ds_ist >= date('{{start_date}}') - interval '90' day
and b.manufacturer_name = 'GCMMF'
group by 1,2
),

grn_agg AS (
    SELECT
        UPPER(iiv.vendor_invoice_id) AS join_invoice_id,
        iiv.po_id as po_number,
        completion_time,
        pg.created_at as grn_created_at,
        CAST(pg.item_id AS VARCHAR) AS join_item_id,
        array_agg(distinct pg.upc_id) as upc_id,
        SUM(pg.quantity) AS grn_quantity,
        SUM(SUM(pg.quantity)) OVER (PARTITION BY UPPER(iiv.vendor_invoice_id), iiv.po_id) as invoice_total_grn_qty,
        avg(landing_price) as landing_price
    FROM ims.ims_inward_invoice iiv
    JOIN po.po_grn pg
        ON pg.grn_id = iiv.grn_id
        AND pg.insert_ds_ist >= cast(date('{{start_date}}') - interval '60' day as varchar)
    WHERE iiv.insert_ds_ist >= cast(date('{{start_date}}') - interval '60' day as varchar)
      AND iiv.lake_active_record
    GROUP BY 1,2,3,4,5
)
,
dn_agg AS (
    with unique_product_upc AS (
        SELECT
            item_id,
            upc
        FROM (
            SELECT
                item_id,
                upc,
                ROW_NUMBER() OVER(PARTITION BY upc ORDER BY id DESC) as rn
            FROM rpc.product_product
        )
        WHERE rn = 1
    )
    SELECT
        UPPER(dn.vendor_invoice_id) AS join_invoice_id,
        dn.po_number,
        CAST(upp.item_id AS VARCHAR) AS join_item_id,
        array_agg(distinct dnpd.upc_id) as upc_id,
        array_agg(distinct dnpd.remark) as all_dn_remarks,
        SUM(quantity) as dn_quantity,
        SUM(SUM(quantity)) OVER (PARTITION BY UPPER(dn.vendor_invoice_id), dn.po_number) as invoice_total_dn_qty,
        sum(net_amount) as net_amount
    FROM pos.discrepancy_note dn
    LEFT JOIN pos.discrepancy_note_product_detail dnpd ON dn.id = dnpd.dn_id_id
    JOIN unique_product_upc upp ON upp.upc = dnpd.upc_id
    WHERE dn.lake_active_record
      AND dnpd.lake_active_record
      AND dn.created_at >= cast(date('{{start_date}}') - interval '60' day as timestamp)
    GROUP BY 1, 2, 3
)
,
asn_check AS (
    SELECT
        UPPER(edi.invoice_id) AS join_invoice_id,
        edi.po_number,
        CAST(edp.item_id AS VARCHAR) AS join_item_id,
        CASE
            WHEN json_extract_scalar(edi.meta, '$.dynamic_threshold_results.asn_verification_status') = 'VERIFIED'
            THEN 'ASN Unloading'
            ELSE 'Non ASN Unloading'
        END AS unloading_type,
        SUM(edp.billed_quantity) AS asn_quantity
    FROM po.edi_integration_partner_po_invoice_mapping edi
    JOIN po.edi_integration_partner_invoice_item_details edp
        ON edi.asn_id = edp.asn_id
    WHERE edi.lake_active_record
      AND edp.lake_active_record
      AND edp.insert_ds_ist >= cast(date('{{start_date}}') - interval '60' day as varchar)
    GROUP BY 1, 2, 3,
        CASE
            WHEN json_extract_scalar(edi.meta, '$.dynamic_threshold_results.asn_verification_status') = 'VERIFIED'
            THEN 'ASN Unloading'
            ELSE 'Non ASN Unloading'
        END
),
calculated_variances AS (
    SELECT
        cid.*,
        completion_time,
        landing_price,
        net_amount,
        grn_created_at,
        COALESCE(ga.grn_quantity, 0) AS grn_quantity,
        COALESCE(da.dn_quantity, 0) AS dn_quantity,
        COALESCE(ga.invoice_total_grn_qty, 0) AS invoice_total_grn_qty,
        COALESCE(da.invoice_total_dn_qty, 0) AS invoice_total_dn_qty,
        COALESCE(ac.unloading_type, 'Non ASN Unloading') AS unloading_type,
        COALESCE(ac.asn_quantity, 0) AS asn_quantity,
        ga.upc_id as grn_upc,
        da.upc_id as dn_upc,
        da.all_dn_remarks,
        (COALESCE(ga.grn_quantity, 0) + COALESCE(da.dn_quantity, 0)) AS total_inward_qty
    FROM core_item_data cid
    LEFT JOIN grn_agg ga
        ON ga.join_invoice_id = cid.join_invoice_id
        AND ga.po_number = cid.po_number
        AND ga.join_item_id = cid.item_id
    LEFT JOIN dn_agg da
        ON da.join_invoice_id = cid.join_invoice_id
        AND da.po_number = cid.po_number
        AND da.join_item_id = cid.item_id
    LEFT JOIN asn_check ac
        ON ac.join_invoice_id = cid.join_invoice_id
        AND ac.po_number = cid.po_number
        AND ac.join_item_id = cid.item_id
    WHERE cid.final_verified_qty IS NOT NULL
      AND cid.final_verified_qty > 0
)
,


payment_data AS (
    SELECT
        ipm.invoice_id,
        vendor_invoice_id,
        SUM(ipm.payment_amount) AS total_payment_value,
        ARRAY_JOIN(ARRAY_AGG(DISTINCT ipd.utr_number), ', ') AS utr_numbers,
        JSON_EXTRACT(bi.meta, '$.tds_amount') AS tds_amount
    FROM po.invoice_payment_mapping ipm
    INNER JOIN po.invoice bi ON bi.id = ipm.invoice_id and bi.active and bi.lake_active_record and bi.insert_ds_ist >= cast(current_date - interval '60' day as varchar)
    LEFT JOIN po.invoice_payment_details ipd
        ON ipd.id = ipm.payment_id
        AND ipd.lake_active_record
    WHERE ipm.lake_active_record
    AND vendor_invoice_id in(select join_invoice_id from calculated_variances group by 1)
    GROUP BY 1,2,5
)

SELECT distinct
    seller_name as entity_name,
    request_id AS vr_id,
    CAST(updated_at as timestamp) as approval_timestamp,
    a.invoice_id,
    a.po_number,
    date(invoice_date) as invoice_date,
    date(grn_date) as grn_date,
    vendor_id,
    a.vendor_name,
    manufacturer_name,
    facility_name,
    city_name,
    a.item_id,
    item_name,
    invoice_landing_price,
    invoice_quantity,
    final_verified_qty,
    asn_quantity,
    po_quantity,
    grn_quantity_purchase_mis as grn_quantity,
    dn_quantity,
    landing_price,
    net_amount,
    c.po_landing_price,
    c.grn_landing_price,
    total_payment_value, utr_numbers, tds_amount,
    gcmmf_customer_code, bank_account
FROM calculated_variances a
left join payment_data b on a.join_invoice_id = b.vendor_invoice_id
left join po_agg c on a.po_number = cast(c.po_number as varchar) and a.item_id = cast(c.item_id as varchar)
left join blinkit_staging.interim.amul_bank_details x on cast(a.vendor_id as varchar) = x.gpos_vendor_id
where manufacturer_name = 'GCMMF'
and vendor_id in(
223056,223073,223393,223539,234241,234247,234248,234249,234250,234253,234254,234316,234317,234319,237138,240933,240944,240979,407175,425457,498727,500776,501225,502481,505220,510103,512210,516467,61911,16009,13860,16004,22374,31480,12059,13655,13276,16711,12968,13125,32495,32993,36914,36937,36938,11481,16847,25448,34625,12297,25596,15063,12342,13704,33534,27138,13337,33198,13082,13440,15232,24987,23721,13574,13945,14011,14862,15181,15322,15724,15924,16782,16815,20194,23371,23372,23578,23729,25732,26842,27276,29859,32992,33269,34624,14796,15643,12373,12458,23544,15398,35019,11587,24421,13041,26798,24308,34631,23537,22740,13569,13072,11283,24738,11194,19446,11052,13039,13653,15144,35020,24735,32366,15635,22331,12971,16231,13452,24944,15303,15318,24529,20859,27986,15647,30204,24902,13439,35771,16458,17635,32416,13844,21554,24900,19547,15720,16003,15518,25523,19246,19615,13211,12465,15360,27902,29620,23093,13371,21127,29585,20422,15552,21666,24585,21310,35784,33127,34046,13369,18958,34748,36450,23094,18403,22299,23240,22282,35549,500475,503946,506274,503888,511340,504723,511162,504390,22586,506269
)
AND date(grn_date) between date'{{start_date}}' and date'{{end_date}}'
'''

gpos_query = render(GPOS_SQL, start_date=START_DATE, end_date=END_DATE)
print(gpos_query)

## 6. Preview each query (small sample)

Pulls a capped sample of each query so you can eyeball the columns/values before the

full extract. Comment this cell out if you only want the full run.

In [ ]:
PREVIEW_ROWS = 5

def preview(query, label):
    sample_sql = f'SELECT * FROM (\n{query}\n) _preview LIMIT {PREVIEW_ROWS}'
    print(f'--- {label} sample ({PREVIEW_ROWS} rows) ---')
    df = sql(sample_sql, label=label)
    return df

hp_preview   = preview(hp_query, 'HPTech')
gpos_preview = preview(gpos_query, 'GPOS')
display(hp_preview)
display(gpos_preview)

## 7. Run both queries -> ONE consolidated Parquet

Each query is streamed in `CHUNKSIZE`-row batches, normalised to the target schema, and

appended to a single `ParquetWriter`. Memory stays bounded to roughly one chunk. If the

driver does not support `chunksize`, it falls back to a single fetch automatically.

In [ ]:
# ============================================================================
# Stream both queries into the consolidated Parquet file
# ============================================================================
def _iter_chunks(query):
    """Yield DataFrame chunks; fall back to one fetch if chunksize unsupported."""
    try:
        for chunk in pd.read_sql_query(sql=query, con=connection, chunksize=CHUNKSIZE):
            yield chunk
    except TypeError:
        yield pd.read_sql_query(sql=query, con=connection)


def stream_to_writer(query, source_label, writer):
    t0, total, preview = time.time(), 0, None
    for chunk in _iter_chunks(query):
        if len(chunk) == 0:
            continue
        norm  = normalize(chunk, source_label)
        writer.write_table(to_table(norm))
        if preview is None:
            preview = norm.head(5).copy()
        total += len(chunk)
        print(f'  [{source_label}] +{len(chunk):>8,}  (total {total:>10,})')
    print(f'  [{source_label}] DONE {time.time()-t0:6.1f}s  {total:,} rows')
    return total, preview


counts = {}
if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)

writer = pq.ParquetWriter(OUT_FILE, TARGET_SCHEMA, compression='snappy')
try:
    counts['HPTech'], hp_head = stream_to_writer(hp_query,   'HPTech', writer)
    counts['GPOS'],   gp_head = stream_to_writer(gpos_query, 'GPOS',   writer)
finally:
    writer.close()

print('\nWrote consolidated parquet:', OUT_FILE)
print('Row counts by source     :', counts, '=> total', sum(counts.values()))

## 8. Optional — consolidated CSV

Streams the Parquet file to CSV in batches (append mode). Skipped automatically when the

row count is above `CSV_MAX_ROWS` or when `WRITE_CSV` is `False`.

In [ ]:
total_rows = pq.ParquetFile(OUT_FILE).metadata.num_rows

if not WRITE_CSV:
    print('WRITE_CSV=False -> skipping CSV.')
elif total_rows > CSV_MAX_ROWS:
    print(f'{total_rows:,} rows > CSV_MAX_ROWS ({CSV_MAX_ROWS:,}) -> skipping CSV. Use the parquet file.')
else:
    t0 = time.time()
    if os.path.exists(CSV_FILE):
        os.remove(CSV_FILE)
    pf = pq.ParquetFile(OUT_FILE)
    header = True
    for batch in pf.iter_batches(batch_size=CHUNKSIZE):
        batch.to_pandas().to_csv(CSV_FILE, mode='a', header=header, index=False)
        header = False
    print(f'Wrote CSV {time.time()-t0:6.1f}s -> {CSV_FILE}')

## 9. Consolidated sample — a few rows from each source

Confirms both HPTech and GPOS rows landed in the single file with aligned columns.

In [ ]:
def sample_by_source(path, per=5):
    """Collect up to `per` rows per source without loading the whole file."""
    pf = pq.ParquetFile(path)
    want = {'HPTech': per, 'GPOS': per}
    frames = []
    for batch in pf.iter_batches(batch_size=CHUNKSIZE):
        pdf = batch.to_pandas()
        for src in list(want):
            if want[src] > 0:
                take = pdf[pdf['source'] == src].head(want[src])
                if len(take):
                    frames.append(take)
                    want[src] -= len(take)
        if all(v <= 0 for v in want.values()):
            break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

consolidated_sample = sample_by_source(OUT_FILE, per=5)
print('Sample rows per source:')
print(consolidated_sample['source'].value_counts().to_string())
display(consolidated_sample)

## 10. Metadata & data-quality checks

Row counts (total + per source), per-column null & blank counts, numeric zero counts,

date-range coverage per source, and on-disk file size. The null/blank pass streams the

file batch-by-batch so it stays memory-safe on large extracts.

In [ ]:
# ============================================================================
# Quality report
# ============================================================================
pf   = pq.ParquetFile(OUT_FILE)
meta = pf.metadata
n    = meta.num_rows
print(f'File            : {OUT_FILE}')
print(f'Total rows      : {n:,}')
print(f'Row groups      : {meta.num_row_groups}')
print(f'On-disk size    : {os.path.getsize(OUT_FILE)/1e6:,.2f} MB')
print()

# ---- streamed accumulation: per-source counts, nulls, blanks, zeros, dates --
src_counts   = {}
null_counts  = {c: 0 for c in COLUMN_ORDER}
blank_counts = {c: 0 for c in STRING_COLS}
zero_counts  = {c: 0 for c in NUM_COLS}
date_span    = {}   # source -> [min_grn, max_grn]

for batch in pf.iter_batches(batch_size=CHUNKSIZE):
    pdf = batch.to_pandas()
    for src, cnt in pdf['source'].value_counts().items():
        src_counts[src] = src_counts.get(src, 0) + int(cnt)
    for c in COLUMN_ORDER:
        null_counts[c] += int(pdf[c].isna().sum())
    for c in STRING_COLS:
        blank_counts[c] += int((pdf[c].fillna('').astype(str).str.strip() == '').sum())
    for c in NUM_COLS:
        zero_counts[c] += int((pdf[c] == 0).sum())
    g = pdf.dropna(subset=['grn_date'])
    for src, sub in g.groupby('source'):
        lo, hi = sub['grn_date'].min(), sub['grn_date'].max()
        if src not in date_span:
            date_span[src] = [lo, hi]
        else:
            date_span[src][0] = min(date_span[src][0], lo)
            date_span[src][1] = max(date_span[src][1], hi)

print('Rows by source  :')
for s, c in sorted(src_counts.items()):
    print(f'    {s:<8} {c:>12,}')
print()
print('grn_date coverage by source:')
for s, (lo, hi) in sorted(date_span.items()):
    print(f'    {s:<8} {str(lo)[:10]} -> {str(hi)[:10]}')
print()

quality = pd.DataFrame({
    'non_null'    : {c: n - null_counts[c] for c in COLUMN_ORDER},
    'null'        : null_counts,
    'null_pct'    : {c: round(100 * null_counts[c] / n, 2) if n else 0 for c in COLUMN_ORDER},
    'blank_str'   : {c: blank_counts.get(c, 0) for c in COLUMN_ORDER},
    'zero_numeric': {c: zero_counts.get(c, 0) for c in COLUMN_ORDER},
}).loc[COLUMN_ORDER]

print('Per-column quality:')
display(quality)

fully_blank = [c for c in COLUMN_ORDER if null_counts[c] + blank_counts.get(c, 0) >= n]
if fully_blank:
    print('\nWARNING - columns that are entirely null/blank:', fully_blank)
else:
    print('\nNo fully-empty columns.')

## 11. Load consolidated file for analysis

From here the parquet is your single input for downstream reconciliation analysis.

In [ ]:
# For moderate data, load fully:
df = pd.read_parquet(OUT_FILE)
print(df.shape)
df.head()

# For very large data, read only what you need, e.g.:
# df = pd.read_parquet(OUT_FILE, columns=['source','invoice_id','vendor_id','net_amount'])
# or scan batch-by-batch via pq.ParquetFile(OUT_FILE).iter_batches(...)

## 12. Reconciliation dashboard

Generates a self-contained HTML dashboard (`amul_recon_dashboard.html`) from the
consolidated parquet — PO vs Invoice/GRN/DN/Net-payable, payment status by invoice,
quantity discrepancies (GRN + DN ≠ invoice), payments due (clean vs open concern),
and city/vendor breakdowns. No external assets, so the file opens anywhere offline.

In [ ]:
# ============================================================================
# Build the reconciliation dashboard (self-contained HTML, no external assets)
# ============================================================================
# Requires build_dashboard.py to sit next to this notebook (same repo folder).
from build_dashboard import build

DASH_FILE = os.path.join(OUT_DIR, 'amul_recon_dashboard.html')
_path, _stats = build(OUT_FILE, DASH_FILE)
print('Dashboard written:', _path)
print('Stats             :', _stats)

# Preview inline (or just open the .html file in a browser for the full view):
from IPython.display import IFrame, display
display(IFrame(src=os.path.relpath(DASH_FILE, start=os.getcwd()), width='100%', height=640))